# Unity Catalog Semantic Layer — Complete Components Guide

**Purpose:** This document provides a comprehensive reference for every Unity Catalog semantic layer component used in the Supply Chain Control Tower Curator Workbench and beyond. For each component, it explains what it is, how to apply it, what it impacts in Unity Catalog, and how it helps Genie (Genie Agents / Genie One / Genie Code) respond with greater accuracy.

**Sources:** All definitions and behaviors are sourced from [Databricks public documentation](https://learn.microsoft.com/en-us/azure/databricks/) (Oct 2026) and validated against the implementation in this project's Curator Workbench notebook.

---

## Architecture Overview: The Genie Ontology

The **Genie Ontology** is the unified context layer that gives the Genie family of AI experiences (Genie One, Genie Code, Genie Agents) a business-aware map of your organization. It brings together two categories of context:

| Category | Description | Examples |
| --- | --- | --- |
| **UC Semantics (Modeled Context)** | Context you define, govern, and certify explicitly through Unity Catalog features | Metric Views, Domains, Pages, Certification, Tags, Comments |
| **Inferred Context** | Context Genie automatically extracts and maintains from your existing assets and usage | Metric definitions from dashboards, authoritative sources from queries, business rules from Genie Agents |

Each inferred snippet gets an **authority score** based on its source, usage frequency, and freshness. Snippets are also gated by UC permissions — Genie only uses snippets extracted from assets you have permission to see.

**Key Principle:** Context you curate once in UC Semantics applies to Genie One, Genie Code, and Genie Agents simultaneously — a single governance model for all AI surfaces.

# CATEGORY A: Unity Catalog Semantics (Modeled Context)

These are the components you explicitly define and govern in Unity Catalog. They form the **human-modeled layer** of the Genie Ontology. Genie treats these as authoritative — higher trust than inferred context.

---

## Component 1: Table & Column Comments

### What It Is
Comments are free-text descriptions attached to Unity Catalog objects (catalogs, schemas, tables, views, materialized views, and individual columns). They are stored as metadata in the metastore and visible in Catalog Explorer, the Information Schema, and to any tool that reads UC metadata.

### How to Apply

```sql
-- Table-level comment
ALTER TABLE catalog.schema.table SET TBLPROPERTIES ('comment' = 'Customer sales orders with fulfillment status');

-- Column-level comment
ALTER TABLE catalog.schema.table ALTER COLUMN order_status COMMENT 'Order fulfillment status: Fulfilled, Partially_Fulfilled, Backordered, Cancelled';

-- AI-generated comments (via Catalog Explorer UI)
-- Click "AI generate" above column headings to auto-suggest comments
```

### What It Impacts in UC
* Stored permanently in the metastore — visible to all users with BROWSE privilege
* Appears in Catalog Explorer, `DESCRIBE TABLE`, `information_schema.columns`, and `information_schema.tables`
* Survives table alterations; must be explicitly removed
* Column comments on views are NOT supported via `ALTER COLUMN COMMENT` (only tables)

### How It Helps Genie
* **Schema Disambiguation:** When two tables have similar column names (e.g., `lead_time_variance_days` in both `supplier_orders` and `supplier_lead_times`), comments tell Genie which table to use for which question
* **Value Enumeration:** Comments listing valid values (e.g., `'Fulfilled, Partially_Fulfilled, Backordered, Cancelled'`) prevent Genie from guessing incorrect WHERE clause values
* **Domain Context:** Comments like "Use ship_date for date filtering, not actual_delivery_date" provide query-writing guidance
* **Accuracy Impact:** In this project, adding column comments fixed wrong-table errors (D04, D06) and status definition confusion (F03, F05, H03)

### Confidence Tier: HEURISTIC
Comments guide Genie's SQL generation heuristically — the LLM reads them and makes better decisions, but the answer is not mathematically guaranteed.

---

## Component 2: Metric Views

### What It Is
Metric Views are a new Unity Catalog object type (`CREATE VIEW ... WITH METRICS LANGUAGE YAML`) that define reusable business KPIs as governed, securable objects. Unlike regular views that return pre-aggregated rows, metric views define **measures** (aggregate expressions) and **fields/dimensions** (grouping attributes) that can be queried at any granularity using the `MEASURE()` function.

### How to Apply

```sql
CREATE OR REPLACE VIEW catalog.schema.delivery_performance_by_region
WITH METRICS LANGUAGE YAML AS
$$
version: 1.1
comment: "Delivery performance KPIs by destination region"
source: catalog.logistics_operations.shipments

filter: source.ship_date BETWEEN DATE '2026-08-01' AND DATE '2026-08-31'

fields:
  - name: Destination Region
    expr: source.destination_region
    comment: "Shipment destination region"

measures:
  - name: on_time_delivery_rate
    expr: ROUND(100.0 * SUM(CASE WHEN source.is_late = false THEN 1 ELSE 0 END) / COUNT(*), 2)
    comment: "Percentage of shipments delivered on time"

  - name: late_delivery_rate
    expr: ROUND(100.0 * SUM(CASE WHEN source.is_late = true THEN 1 ELSE 0 END) / COUNT(*), 2)
    comment: "Percentage of late shipments"

  - name: avg_delay_days
    expr: ROUND(AVG(CASE WHEN source.is_late = true THEN source.delay_days END), 2)
    comment: "Average delay in days for late shipments"
$$
```

### What It Impacts in UC
* Metric views are **first-class UC securable objects** — governed by standard UC privileges (GRANT/REVOKE)
* Queryable with `SELECT MEASURE(measure_name) FROM metric_view WHERE ...`
* The YAML definition enforces a single source of truth for business metric calculations
* Supports joins, filters, window measures, and cross-references between measures
* Visible in Catalog Explorer with dedicated metric view UI

### How It Helps Genie
* **DETERMINISTIC Answers:** Instead of generating SQL from scratch (which can vary across runs), Genie queries the pre-defined metric view directly — the calculation is locked
* **Eliminates SQL Variance:** Every time any user asks "what is the on-time delivery rate?", the exact same aggregation logic executes
* **Cross-Reuse:** The same metric view serves Genie Agents, Genie One, dashboards, and notebooks
* **Accuracy Impact:** In this project, metric views converted 25+ tests from INFERRED/GUESSED to DETERMINISTIC confidence tier

### Confidence Tier: DETERMINISTIC
When Genie uses a metric view, the answer is mathematically locked to the pre-defined formula. This is the highest confidence tier.

---

## Component 3: Domains & Subdomains

### What It Is
Domains are a business-aligned organization layer in Unity Catalog that groups data assets (catalogs, schemas, tables, dashboards, Genie Agents) so people can browse and find data by **business purpose** rather than technical structure. They appear on the **Discover** page.

### How to Apply
* Currently **UI-only** (no API): Created via the Discover page in the Databricks workspace
* You create a Domain (e.g., "Supply Chain Operations"), then add schemas, tables, or other assets to it
* Subdomains can be nested within domains for finer organization
* Assets can also be assigned to domains via governed tags

### What It Impacts in UC
* Provides a **business taxonomy** layered on top of the technical catalog/schema hierarchy
* Does NOT affect permissions, storage, or data — purely organizational metadata
* Deleting a domain does NOT delete schemas or tables (they belong to the catalog)
* Assets can belong to multiple domains

### How It Helps Genie
* **Asset Discovery:** Genie One uses domain structure to understand which assets are related to a business area
* **Contextual Grouping:** When a user asks a supply chain question, the domain tells Genie that logistics, inventory, supplier, and demand schemas all belong to the same business context
* **Navigation:** Business users on the Discover page find relevant data faster
* **Accuracy Impact:** Domains help Genie understand organizational boundaries, reducing cross-domain confusion

---

## Component 4: UC Pages

### What It Is
Pages are **governed knowledge documents** within Unity Catalog that define business concepts, terms, policies, and rules as an authoritative source. They are the UC equivalent of a business glossary or data dictionary. Both humans and Genie One reference them.

### How to Apply
* Currently **UI-only** (no API): Created within Domains on the Discover page
* Each Page has a title, rich-text body, and can link to **Related Assets** (tables, views, metric views)
* Pages are organized within Domains

### Example Pages from This Project

| Page | Domain | Content |
| --- | --- | --- |
| Fiscal Calendar & Targets | Supply Chain Operations | Defines fiscal year (Jul-Jun), Q1 = Jul-Sep, Q1 target = 92% service level |
| Cross-Domain Metric Definitions | Supply Chain Operations | Defines Cost of Disruption formula, Revenue at Risk components |
| Logistics Risk Standards | Logistics Operations | Defines "on-time" vs "late" thresholds, wasted freight calculation |
| Demand Quality Standards | Demand Analysis | Defines fulfillment rate formula, order status hierarchy |
| Inventory Risk Classification | Inventory Management | Defines safety stock thresholds, stockout classification |
| Supplier Quality Standards | Supplier Procurement | Defines vendor late rate calculation, per-order vs per-vendor |
| Executive Alert Thresholds | Executive Reporting | Defines critical/warning/acceptable thresholds for KPIs |

### What It Impacts in UC
* Governed by UC permissions — only authorized users can create/edit
* Linked Related Assets create a navigable relationship graph
* Pages persist independently of tables/views — they are organizational knowledge, not data

### How It Helps Genie
* **Business Term Resolution:** When a user asks "Are we going to miss our Q1 targets?", Genie One reads the Fiscal Calendar Page to learn Q1 = Jul-Sep and the target = 92%
* **Policy Disambiguation:** The Supplier Quality Standards Page resolves whether "vendor late rate" means per-order or per-vendor
* **Authoritative Knowledge:** Pages are treated as high-authority sources in the Genie Ontology
* **Accuracy Impact:** In this project, UC Pages were the ONLY mechanism that could resolve fiscal calendar questions (G01, G02) and policy threshold questions (P01-P05)

### Confidence Tier: DETERMINISTIC (when the Page content is precise enough to lock the answer)

---

## Component 5: Governed Tags

### What It Is
Tags are key-value attributes applied to Unity Catalog securable objects (tables, columns, schemas, catalogs) to organize and categorize them. **Governed tags** are a special class where admins define the allowed keys and values centrally — ensuring consistent tagging across the catalog.

### How to Apply

```sql
-- Table-level tags
ALTER TABLE catalog.schema.shipments SET TAGS ('domain' = 'logistics', 'data_tier' = 'silver');

-- Column-level tags
ALTER TABLE catalog.schema.customers ALTER COLUMN ssn SET TAGS ('pii' = 'ssn');

-- Schema-level tags
ALTER SCHEMA catalog.logistics_operations SET TAGS ('business_domain' = 'logistics');

-- Query tags via information_schema
SELECT * FROM information_schema.table_tags WHERE tag_name = 'domain';
SELECT * FROM information_schema.column_tags WHERE tag_name = 'pii';
```

### Constraints
* Max 50 tags per securable object; max 1,000 column tags per table
* Tag keys are case-sensitive; max 256 chars for keys and values
* Characters `. , - = / :` are not allowed in tag keys
* Cannot assign tags to multiple columns in a single ALTER TABLE command

### What It Impacts in UC
* Tags are searchable metadata — users can filter assets by tag in Catalog Explorer
* Governed tags enforce a controlled vocabulary across the organization
* Tags can be used for ABAC (Attribute-Based Access Control) policies
* Tags can assign assets to Domains automatically

### How It Helps Genie
* **Domain Routing:** Schema-level tags (e.g., `'business_domain' = 'logistics'`) help Genie understand which schemas serve which business areas
* **Data Classification:** Tags like `'data_tier' = 'gold'` or `'certified' = 'true'` signal which tables are production-ready
* **Asset Prioritization:** Genie can prefer tagged/governed assets over untagged ones
* **Accuracy Impact:** In this project, governed tags helped Genie correctly route questions to the right domain agent

---

## Component 6: Certification & Deprecation

### What It Is
Certification and deprecation are system-level signals (implemented via system tags) that mark a data asset as **trusted** (certified) or **outdated** (deprecated). A blue checkmark appears next to certified assets in the workspace.

### How to Apply
* Applied via Catalog Explorer UI or by setting system tags
* Certification signals that the asset meets your organization's quality standards
* Deprecation warns that the asset is outdated and should not be used

### What It Impacts in UC
* Visible indicators in Catalog Explorer and search results
* Influences search ranking — certified assets surface higher
* Part of UC Semantics alongside metric views, domains, and pages

### How It Helps Genie
* **Trust Signal:** Genie One prioritizes certified assets when generating answers
* **Steering:** Certification steers Genie toward the assets your organization vouches for
* **Avoidance:** Deprecated assets are de-prioritized, reducing the chance of using stale data
* **Accuracy Impact:** Ensures Genie prefers the authoritative version of a table or view when duplicates exist

# CATEGORY B: Genie Agent Tuning Components

These components are configured within individual Genie Agents (formerly Genie Spaces) to improve response accuracy. They are agent-scoped — changes apply only to the specific agent where they are configured, unlike UC Semantics which apply globally.

---

## Component 7: Example SQL Queries (Certified/Trusted Queries)

### What It Is
Example SQL queries are pre-written SQL statements paired with natural-language question titles, added to a Genie Agent's **Examples tab**. When a user asks a question that matches (or is similar to) an example, Genie can use the query directly or learn from it to handle similar questions.

### How to Apply
* **UI:** Agent Settings → Configure → Examples tab → Add → Example Query
* **API:** Via the `example_question_sqls` field in the Genie Agent's serialized space configuration
* Each example has:
  * **Title:** The typical user phrasing (e.g., "What percentage of vendors had delayed deliveries last month?")
  * **SQL:** The verified query that answers it
* Can be static or parameterized

### Limits
* Up to **100 instructions** per agent (example queries + SQL functions + general instructions text all count toward this limit)

### What It Impacts
* Scoped to the individual Genie Agent — does not affect other agents or Genie One
* Saved queries appear as **"Trusted queries"** in user responses (with a trust badge)
* Users with CAN EDIT can view the SQL used

### How It Helps Genie
* **Direct Match:** When the user's question closely matches an example title, Genie uses the pre-written SQL directly — no SQL generation needed
* **Pattern Learning:** Even for non-exact matches, Genie learns query patterns (JOIN strategies, WHERE clause patterns, aggregation approaches) from examples
* **Disambiguation:** Resolves ambiguous business terms (e.g., "vendor late rate" = per-order percentage, not per-vendor)
* **Accuracy Impact:** In this project, example queries fixed F02 (vendor late ambiguity) and F06 (worst product family decline)

### Confidence Tier: HEURISTIC
Genie may use the example directly (high confidence) or adapt it (moderate confidence).

---

## Component 8: SQL Functions (Trusted Functions)

### What It Is
Unity Catalog SQL functions (scalar or table-valued) registered as data sources on a Genie Agent. When a question involves complex logic that cannot be captured with a static query, Genie calls the function with user-supplied parameters. Genie **cannot view or modify** the function's internal SQL — it's a black box.

### How to Apply

```sql
-- Create a SQL function in Unity Catalog
CREATE OR REPLACE FUNCTION catalog.schema.check_on_time_delivery_threshold(
    region STRING,
    month_start DATE
) RETURNS TABLE (
    region STRING,
    on_time_rate DOUBLE,
    threshold DOUBLE,
    status STRING
)
COMMENT 'Checks if on-time delivery rate meets the critical threshold for a given region and month.'
RETURN
    SELECT ...
```

* **Register on Agent:** Agent Settings → Sources → SQL function → Add function
* Functions must be registered per-agent (not automatic from UC)

### What It Impacts
* Functions are stored in UC — governed, versioned, shareable across agents and teams
* Appears as **"Trusted function"** in user responses
* Genie shows the function's COMMENT to users, not the internal SQL

### How It Helps Genie
* **Encapsulated Logic:** Complex threshold checks, multi-step calculations, or business rules that are too intricate for example queries
* **Black-Box Accuracy:** The function's logic is locked — Genie cannot misinterpret or rewrite it
* **Parameter Passing:** Genie infers parameter values from the user's question and passes them correctly
* **Accuracy Impact:** In this project, 5 threshold functions (P01-P05) made policy-threshold answers DETERMINISTIC

### Confidence Tier: DETERMINISTIC
Function output is computed, not generated — the answer is mathematically locked.

---

## Component 9: Plain-Text Instructions (General Instructions)

### What It Is
Free-text guidance added to a Genie Agent's **Text tab** that tells Genie how to interpret questions, format responses, and apply domain-specific rules globally across all prompts.

### How to Apply
* **UI:** Agent Settings → Configure → Text tab
* **API:** Via the `instructions.text_instructions` field in the serialized space
* Instructions should be formatted as structured Markdown with bullet points

### Best Practices (from Databricks docs)
* Use SQL expressions and example queries first; text instructions only as a last resort
* Keep instructions focused and non-conflicting
* Good for: fiscal year definitions, formatting rules, language preferences, date context
* Bad for: metric definitions or query patterns (use SQL expressions or examples instead)

### Example from This Project
```
SAFEGUARDS:
- If a question is outside your domain, say 'This question is outside my area of expertise'.
- Do not guess or fabricate data.
- Date context: 'Last month' = August 2026. 'Prior month' = July 2026.
- Always use explicit date literals (e.g. DATE '2026-08-01'). Never use CURRENT_DATE().
- Region values: Western, Eastern, Central, Southern. 'West' maps to 'Western'.
```

### How It Helps Genie
* **Global Context:** Applied to every question, ensuring consistent date interpretation, region mapping, and response format
* **Guardrails:** Prevents Genie from guessing outside its domain or fabricating data
* **Value Mapping:** Explicit mapping of user terms to data values (e.g., "West" → "Western")
* **Accuracy Impact:** Prevents date-related errors and cross-domain hallucination

### Confidence Tier: INFERRED
Instructions are interpreted by the LLM — helpful but not guaranteed.

---

## Component 10: Knowledge Store (SQL Expressions: Measures, Filters, Fields)

### What It Is
The Knowledge Store is a collection of curated semantic definitions within a Genie Agent. It includes three types of SQL expressions that teach Genie about common business terms:

| Type | Purpose | Example |
| --- | --- | --- |
| **Measures** | KPI calculations (aggregate expressions) | `Win rate = COUNT(CASE WHEN stage = 'Closed Won' THEN 1 END) / NULLIF(COUNT(*), 0)` |
| **Filters** | Common filtering conditions | `High-value orders = orders.amount > 10000` |
| **Fields** | Attributes for grouping/analyzing | `Deal size = CASE WHEN amount < 10000 THEN 'Small' WHEN amount < 50000 THEN 'Medium' ELSE 'Large' END` |

### How to Apply
* **UI:** Agent Settings → Configure → Examples tab → Add → Filter / Measure / Field
* Each expression has: **Name**, **Code** (SQL), **Synonyms**, and **Instructions**
* Synonyms are critically important — they map colloquial terms to the formal definition

### Limits
* Up to **200 knowledge store snippets** per agent (table descriptions + join relationships + SQL expressions share this limit)

### How It Helps Genie
* **Structured Definitions > Text Instructions:** SQL expressions produce more consistent results than text instructions because Genie applies the logic exactly as written
* **Synonym Resolution:** Users can ask about "close rate" or "conversion rate" and Genie maps it to the `Win rate` measure
* **Reusable Logic:** The same measure definition applies across all questions in the agent
* **Accuracy Impact:** SQL expressions provide precision that text instructions cannot

---

## Component 11: Join Relationships

### What It Is
Explicit definitions of how tables in a Genie Agent relate to each other, stored in the Knowledge Store. Primary and foreign keys defined in your UC schema are **automatically** saved as join relationships.

### How to Apply
* **Automatic:** UC primary/foreign keys → auto-imported as join relationships
* **Manual (UI):** Agent Settings → Configure → Define join relationships
* **Knowledge Mining:** Genie can suggest new join relationships based on author interactions (thumbs-up, query downloads)

### How It Helps Genie
* **Correct JOINs:** Prevents Genie from generating incorrect or cartesian joins
* **Multi-Table Queries:** Enables Genie to answer questions that span multiple tables
* **Consistency:** Ensures the same join path is used every time

---

## Component 12: Benchmark Questions

### What It Is
A set of test questions (up to 500 per agent) that you run to assess Genie's overall response accuracy. Each benchmark has a question, an optional SQL answer (ground truth), and an optional evaluation note.

### How to Apply
* **UI:** Agent page → Benchmarks tab → Add question
* **API:** Via the `benchmarks` field in the serialized space
* **Run:** Click "Run benchmarks" to execute all or a subset of questions
* Two modes:
  * **Chat mode:** Compares Genie's SQL-generated results against provided SQL answer
  * **Agent mode:** Multi-step reasoning with LLM judge grading

### What It Impacts
* Benchmark results are timestamped — you can track accuracy over time as you refine the agent
* Results show: accuracy score, execution status, and per-question pass/fail
* You can use Genie Code to analyze a benchmark run and suggest improvements

### How It Helps Genie
* **Quality Gate:** Validates that agent changes (new comments, examples, instructions) actually improve accuracy before publishing
* **Regression Detection:** Catch regressions when adding new instructions that conflict with existing ones
* **Iterative Refinement:** The benchmark → fix → re-benchmark cycle is the primary workflow for tuning agent quality
* **Accuracy Impact:** In this project, benchmarks provided the automated scoring that measured improvement across 3 iterations

---

## Component 13: Entity Matching

### What It Is
An advanced column feature in Genie Agents that enables fuzzy matching of user-supplied values to actual column values. When enabled on a string column, Genie can match approximate inputs to exact data values.

### How to Apply
* **UI:** Agent Settings → Column Settings → Enable entity matching on specific string columns
* **API:** Via the Genie Agent configuration API (entity matching flags per column)
* Each agent supports entity matching on up to **120 string columns**

### Example
* User says: "West region" → Entity matching maps to actual value: "Western"
* User says: "online" → Entity matching maps to: "Online"
* User says: "home goods" → Entity matching maps to: "Home Goods"

### How It Helps Genie
* **Value Resolution:** Users rarely know the exact casing, spelling, or format of data values
* **WHERE Clause Accuracy:** Genie generates `WHERE region = 'Western'` instead of `WHERE region = 'West'` (which would return no results)
* **Natural Language Tolerance:** Supports colloquial or abbreviated user inputs
* **Accuracy Impact:** In this project, entity matching fixed region-name mismatches across multiple test groups

---

## Component 14: Column Metadata (Agent-Scoped)

### What It Is
In addition to UC-level column comments, Genie Agents support **agent-scoped column metadata** — custom descriptions and synonyms that apply only within that agent. This metadata does NOT overwrite Unity Catalog metadata.

### How to Apply
* **UI:** Agent Settings → Data Sources → Click a table → Edit column metadata
* Add descriptions and synonyms per column

### How It Helps Genie
* **Agent-Specific Context:** Different agents can interpret the same column differently based on their domain
* **Synonym Support:** Map colloquial terms to column names (e.g., "revenue" → `total_amount`)
* **Non-Destructive:** Does not alter the UC metadata visible to other tools

# CATEGORY C: Inferred Context (Genie Ontology)

These components are **automatically generated and maintained** by Genie based on your existing assets and usage patterns. You can curate and refine them, but they start as system-inferred.

---

## Component 15: Knowledge Snippets (Genie Ontology Inferred Context)

### What It Is
Knowledge snippets are automatically extracted pieces of business context that Genie maintains from your existing assets: metric views, dashboards, SQL queries, and Genie Agents. Each snippet gets an **authority score** based on its source, usage frequency, and freshness.

### Types of Inferred Snippets

| Type | Example |
| --- | --- |
| **Metric Definitions** | "An 'active user' is a distinct user, deduplicated across all platforms." |
| **Authoritative Sources** | "Revenue questions should be answered using the curated Finance Genie Agent." |
| **Business Rules** | "A 'qualified lead' only counts once a demo is booked." |

### How They Work
* Genie automatically extracts and maintains snippets from your existing assets
* Snippets are gated by UC permissions — Genie only uses snippets from assets you can see
* When you ask a question, Genie ranks relevant snippets, resolves conflicts, and answers using permitted sources
* Genie One and Genie Code search the **same ontology** — context curated once applies to both

### How to Curate
* **Accept:** When Genie suggests a snippet via knowledge mining, accept it to increase its authority
* **Edit:** Refine snippet text for precision
* **Reject:** Remove incorrect or misleading snippets
* At this time, customers do not incur costs for curating ontology snippets

### How It Helps Genie
* **Shared Context:** Business definitions extracted from one dashboard can improve answers across Genie One, Genie Code, and other agents
* **Authority Ranking:** Higher-authority snippets (from certified assets, frequently used queries) take precedence
* **Automatic Maintenance:** Snippets update as your assets evolve

---

## Component 16: Knowledge Mining Recommendations

### What It Is
An automatic system within Genie Agents that analyzes UC metadata and author interactions to suggest improvements to the Knowledge Store.

### How It Works
1. **Metadata Analysis:** Genie analyzes UC metadata for connected tables — primary/foreign keys are auto-saved as join relationships
2. **Interaction Learning:** When an author thumbs-up a response or downloads results, Genie analyzes the query and suggests:
   * New SQL expressions (measures, filters, fields)
   * Additional join relationships
3. **Recommendation Review:** The agent author reviews and accepts/rejects suggestions

### How It Helps Genie
* **Continuous Improvement:** The agent gets smarter with usage without manual curation effort
* **Pattern Discovery:** Surfaces business logic patterns that the author may not have thought to define explicitly
* **Reduced Maintenance:** Automates the most common Knowledge Store updates

# CATEGORY D: Supporting Data Components

These are standard UC objects that serve as part of the semantic layer when combined with the components above.

---

## Component 17: Reference Tables

### What It Is
Standard Unity Catalog tables that contain reference or lookup data which agents query for context that doesn't fit in metadata (comments, tags) or configuration (instructions). They are queryable data that backs governance definitions.

### Example from This Project
```sql
CREATE TABLE catalog.reporting.fiscal_targets (
    fiscal_quarter STRING,
    quarter_months STRING,
    kpi_name STRING,
    target_value DOUBLE,
    target_unit STRING
);

-- Contains rows like:
-- ('Q1', 'Jul-Aug-Sep', 'service_level_pct', 92.0, 'percent')
-- ('Q2', 'Oct-Nov-Dec', 'service_level_pct', 94.0, 'percent')
```

### How It Helps Genie
* **Queryable Context:** Unlike UC Pages (prose) or instructions (text), reference tables provide structured data that Genie can JOIN to or query with SQL
* **Dynamic Values:** Targets, thresholds, and policies that change over time live in tables, not hardcoded in instructions
* **Cross-Agent Sharing:** Any agent with the table in its data sources can access the same reference data
* **Accuracy Impact:** In this project, `fiscal_targets` resolved Q1 target questions (G01, G02) that no other mechanism could answer

---

## Component 18: Reporting Views (Standard SQL Views)

### What It Is
Standard `CREATE VIEW` objects in Unity Catalog that pre-join or pre-aggregate data from base tables into domain-specific reporting shapes. Unlike metric views, these are traditional SQL views — they return pre-determined columns and rows.

### Example from This Project
* `executive_kpis` — pre-aggregated executive KPI summary
* `cost_of_disruption` — cross-domain view joining demand + logistics + inventory for CoD calculation

### How It Helps Genie
* **Simplified Schema:** Genie has fewer tables to reason about — the view encapsulates complex joins
* **Cross-Domain Answers:** Views that join data across schemas (e.g., demand + logistics) enable Genie to answer questions no single base table can
* **Consistent Logic:** The view's SQL is locked — Genie queries it like a table
* **Accuracy Impact:** In this project, the Cost of Disruption view enabled cross-domain tests (E03, H05, H07) that no single-domain agent could answer

---

## Component 19: Supervisor Agent Architecture

### What It Is
A Supervisor Agent is a multi-agent orchestration layer that sits above individual domain Genie Agents. It receives a user question, routes it to the appropriate domain agent, and returns the answer. The Supervisor can also have SQL functions as tools.

### Architecture Patterns Tested in This Project

| Pattern | Tools | Step Budget | Strengths |
| --- | --- | --- | --- |
| **Domain Supervisor** | 5 Genie Agents + 5 SQL functions | 15 steps (1 per genie_space call) | Full routing control, high accuracy on individual questions |
| **MCP Supervisor** | 1 genie_one_mcp tool | 15 steps (3 per MCP round-trip = max 5 calls) | Access to UC Pages, ontology, knowledge snippets via Genie One |
| **Genie One (standalone)** | Full ontology, UC Pages, snippets | N/A | Best for comprehensive prompts, automatic ontology access |

### How It Helps Genie
* **Question Routing:** The Supervisor decides which domain agent is best for each question
* **Multi-Domain Answers:** For cross-domain questions, the Supervisor can call multiple agents
* **Tool Access:** SQL functions as Supervisor tools provide deterministic threshold checks

# Confidence Tiers & Progressive Iteration Strategy

## The Four Confidence Tiers

Every answer from a Genie Agent can be classified into one of four confidence tiers based on **how** the answer was derived:

| Tier | Icon | Meaning | Repeatability | Production-Ready? |
| --- | --- | --- | --- | --- |
| **DETERMINISTIC** | 🟢 | Answer locked by a governed UC asset (metric view, SQL function, certified query) | 100% — same answer every time | Yes |
| **HEURISTIC** | 🟡 | Answer guided by column comments, example queries, or instructions | High — usually correct but may vary | Conditional |
| **INFERRED** | 🟠 | Genie inferred the answer from raw schema + general SQL knowledge | Moderate — may change across runs | No |
| **GUESSED** | 🔴 | Genie guessed with minimal context — often wrong table, wrong filter, or fabricated | Low — unreliable | No |

**The goal of the semantic layer is to move every business-critical metric from GUESSED/INFERRED to DETERMINISTIC.**

---

## Progressive Iteration Strategy (from This Project)

The Curator Workbench demonstrates a 3-iteration approach where each iteration adds more semantic context, progressively improving accuracy:

### Iteration 0: Baseline (Bare Schema)
* **UC Features Used:** None — just raw tables with no metadata
* **What Genie Has:** Table names, column names, data types only
* **Result:** Genie must guess everything — which table, which column, which filter values
* **Typical Issues:** Wrong table selection, incorrect status values, no date context

### Iteration 1: Column Comments + Example SQL + Benchmarks
* **UC Features Used:** `ALTER COLUMN COMMENT`, `ALTER TABLE SET TBLPROPERTIES`, Example SQL Queries (via API), Benchmark Questions (via API), Plain-text Instructions
* **What It Fixes:** Table disambiguation, status value confusion, domain boundaries
* **Confidence Shift:** GUESSED → HEURISTIC for most tests
* **Key Insight:** Comments + examples handle 80% of common errors but cannot lock calculations

### Iteration 2: Metric Views + Governed Tags + Cross-Domain Views
* **UC Features Used:** `CREATE VIEW WITH METRICS LANGUAGE YAML`, `ALTER TABLE SET TAGS`, `ALTER SCHEMA SET TAGS`, Cross-domain reporting views
* **What It Fixes:** Calculation variance, cross-domain answers, metric standardization
* **Confidence Shift:** HEURISTIC → DETERMINISTIC for all metric view measures
* **Key Insight:** Metric views are the single most impactful component — they eliminate SQL generation variance entirely

### Iteration 3: Reference Tables + SQL Functions + UC Pages + Entity Matching
* **UC Features Used:** Reference tables (`fiscal_targets`), SQL Functions (5 threshold functions), UC Domains + UC Pages (manual), Entity Matching (API), ai_forecast instructions
* **What It Fixes:** Fiscal calendar questions, policy threshold questions, region name fuzzy matching
* **Confidence Shift:** Remaining HEURISTIC → DETERMINISTIC for all edge cases
* **Key Insight:** UC Pages are the only mechanism for business glossary / policy definitions; SQL functions lock threshold logic

---

## Summary: Which Component Solves Which Problem?

| Problem Category | Root Cause | Best Solution | Confidence Achieved |
| --- | --- | --- | --- |
| Wrong table selected | Ambiguous column names across tables | Column Comments | HEURISTIC |
| Wrong filter values | Unknown status/category values | Column Comments + Entity Matching | HEURISTIC |
| Inconsistent KPI calculation | LLM generates different SQL each time | Metric Views | DETERMINISTIC |
| Cross-domain questions | No single table has all needed data | Reporting Views + CoD View | DETERMINISTIC |
| Business term ambiguity | "Vendor late rate" means different things | Example SQL Queries + UC Pages | HEURISTIC / DETERMINISTIC |
| Fiscal calendar context | Q1 = Jul-Sep is org-specific knowledge | UC Pages + Reference Tables | DETERMINISTIC |
| Policy thresholds | "Critical" means different things per domain | SQL Functions + UC Pages | DETERMINISTIC |
| Region name mismatch | User says "West", data has "Western" | Entity Matching + Instructions | HEURISTIC |
| Date context | "Last month" is ambiguous | Plain-text Instructions | INFERRED |
| Data classification | Which tables are production-ready? | Governed Tags + Certification | N/A (routing signal) |
| Asset discoverability | Users can't find the right data | Domains + Pages + Tags | N/A (discovery signal) |

---

## Component Quick-Reference by Scope

| Scope | Components |
| --- | --- |
| **Global (UC-wide)** | Comments, Metric Views, Domains, Pages, Governed Tags, Certification, Reference Tables, Views, SQL Functions |
| **Agent-Scoped** | Example SQL Queries, Benchmarks, Instructions, Knowledge Store (Measures/Filters/Fields), Join Relationships, Entity Matching, Column Metadata |
| **Auto-Inferred** | Knowledge Snippets, Knowledge Mining Recommendations |

---

## Documentation References

| Topic | Link |
| --- | --- |
| UC Semantics Overview | [learn.microsoft.com/azure/databricks/uc-semantics](https://learn.microsoft.com/en-us/azure/databricks/uc-semantics/index/) |
| Metric Views | [learn.microsoft.com/azure/databricks/uc-semantics/metric-views/create](https://learn.microsoft.com/en-us/azure/databricks/uc-semantics/metric-views/create/) |
| Domains & Subdomains | [learn.microsoft.com/azure/databricks/uc-semantics/domains](https://learn.microsoft.com/en-us/azure/databricks/uc-semantics/domains/) |
| Genie Ontology | [learn.microsoft.com/azure/databricks/genie/genie-ontology](https://learn.microsoft.com/en-us/azure/databricks/genie/genie-ontology/) |
| Tune Genie Agent Quality | [learn.microsoft.com/azure/databricks/genie-agents/tune-quality](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/tune-quality/) |
| Genie Agent Best Practices | [learn.microsoft.com/azure/databricks/genie-agents/best-practices](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/best-practices/) |
| Test & Monitor Genie Agent | [learn.microsoft.com/azure/databricks/genie-agents/monitor](https://learn.microsoft.com/en-us/azure/databricks/genie-agents/monitor/) |
| Apply Tags to UC Objects | [learn.microsoft.com/azure/databricks/database-objects/tags](https://learn.microsoft.com/en-us/azure/databricks/database-objects/tags/) |
| Certification & Deprecation | [learn.microsoft.com/azure/databricks/data-governance/unity-catalog/certify-deprecate-data](https://learn.microsoft.com/en-us/azure/databricks/data-governance/unity-catalog/certify-deprecate-data/) |
| AI-Generated Comments | [learn.microsoft.com/azure/databricks/comments/ai-comments](https://learn.microsoft.com/en-us/azure/databricks/comments/ai-comments/) |